# Lesson 13: Short-Term Memory — Remember the Last Result

## Objective
Use `MessagesState` and conversation history to give the arithmetic agent **short-term memory**: it can reference the previous result in the same session.

## Limitation of Lesson 12
- ❌ Each `invoke()` was stateless — agent forgot everything after each call
- ❌ "What is double the last answer?" would fail — no context
- ❌ No conversation history

## What is NEW in Lesson 13?
- ✅ `MessagesState` — LangGraph's built-in state class with message accumulation
- ✅ `add_messages` reducer — the special reducer for `HumanMessage`/`AIMessage` lists
- ✅ Short-term memory pattern: pass the growing message list to the LLM
- ✅ `HumanMessage`, `AIMessage` message types
- ✅ The agent can now reference previous results

## Theory: MessagesState

```python
from langgraph.graph import MessagesState
# Equivalent to:
class MessagesState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
```

`add_messages` is a special reducer that:
1. **Appends** new messages to the list
2. **Deduplicates** by message ID (prevents double-adding)
3. Handles both HumanMessage and AIMessage

The entire conversation history lives in `state["messages"]`.  
Passing the full message list to the LLM gives it "memory" of the conversation.

### Short-term vs Long-term Memory
| Type | Scope | Storage |
|------|-------|---------|
| Short-term | Current session only | In-memory (messages list) |
| Long-term | Across sessions | Database / vector store (Lesson 15) |


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Optional, Annotated
from langgraph.graph.message import add_messages
from IPython.display import Image, display

load_dotenv()

vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)

llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

print("✓ Setup complete")
print("  MessagesState: built-in state with add_messages reducer")
print("  add_messages:  special reducer that appends + deduplicates messages")


In [ ]:
# ── Cell 2: Custom State Extending MessagesState ──────────────────────────────
# MessagesState already has: messages: Annotated[list[BaseMessage], add_messages]
# We extend it with our arithmetic fields

class ArithmeticState(MessagesState):
    last_result: Optional[float]   # Remembered between turns
    session_log: list              # Human-readable session history

print("✓ ArithmeticState defined (extends MessagesState)")
print("  Inherited: messages (with add_messages reducer)")
print("  Added: last_result, session_log")


In [ ]:
# ── Cell 3: Arithmetic Agent with Memory ─────────────────────────────────────
SYSTEM_PROMPT = """You are an arithmetic assistant with memory. You can remember the last result.

Rules:
- For arithmetic questions like "what is X plus Y?", compute and answer
- If the user says "double that" or "add 5 to the result", use the last result
- Always end your response with: RESULT: <number>
- Keep responses brief

Examples:
User: "what is 6 plus 4?"    → "6 + 4 = 10. RESULT: 10"
User: "multiply 3 by 7"      → "3 × 7 = 21. RESULT: 21"
User: "double that"          → "Previous result was X. X × 2 = 2X. RESULT: 2X"
"""

def arithmetic_agent(state: ArithmeticState) -> dict:
    messages = state["messages"]
    last_result = state.get("last_result")
    
    # Build context message if we have a previous result
    context_msgs = [SystemMessage(content=SYSTEM_PROMPT)]
    if last_result is not None:
        context_msgs.append(SystemMessage(
            content=f"Note: The last computed result was {last_result}"
        ))
    
    # Add full conversation history (this IS the memory)
    context_msgs.extend(messages)
    
    print(f"  [agent] Processing with {len(messages)} messages in history")
    
    response = llm.invoke(context_msgs)
    ai_content = response.content.strip()
    print(f"  [agent] Response: {ai_content}")
    
    # Extract RESULT: number from response
    result = last_result  # Default: keep last result
    for line in ai_content.split("\n"):
        if "RESULT:" in line:
            try:
                result = float(line.split("RESULT:")[1].strip())
            except:
                pass
    
    log_entry = f"Q: {messages[-1].content} → A: {ai_content}"
    
    return {
        "messages": [AIMessage(content=ai_content)],  # add_messages appends this
        "last_result": result,
        "session_log": [log_entry]
    }

print("✓ Arithmetic agent with memory defined")


In [ ]:
# ── Cell 4: Build Graph ──────────────────────────────────────────────────────
builder = StateGraph(ArithmeticState)
builder.add_node("agent", arithmetic_agent)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

graph = builder.compile()
print("✓ Graph compiled")


In [ ]:
# ── Cell 5: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 6: Simulate a Multi-Turn Session ────────────────────────────────────
# We manually maintain message history between turns
# (Lesson 17 shows how checkpointing does this automatically)

print("Starting arithmetic session with memory:")
print("="*55)

# Session state — persists between turns
session_state = {
    "messages": [],
    "last_result": None,
    "session_log": []
}

def ask(question: str):
    global session_state
    print(f"\nUser: {question}")
    
    # Add user message to state
    session_state["messages"] = session_state["messages"] + [HumanMessage(content=question)]
    
    # Run the graph
    out = graph.invoke(session_state)
    
    # Update session state with new messages + result
    session_state = out
    
    print(f"Agent: {out['messages'][-1].content}")
    print(f"  [last_result = {out['last_result']}]")
    return out

# Turn 1: Fresh question
ask("what is 8 plus 7?")


In [ ]:
# Turn 2: Reference previous result
ask("now multiply that by 2")


In [ ]:
# Turn 3: Another reference
ask("subtract 5 from the last result")


In [ ]:
# Turn 4: Fresh calculation
ask("what is 12 divided by 4?")


In [ ]:
# ── Cell 7: Inspect the Full Conversation History ─────────────────────────────
print("\n── Full Conversation History ─────────────────────")
for i, msg in enumerate(session_state["messages"]):
    role = "User" if isinstance(msg, HumanMessage) else "Agent"
    print(f"  [{i+1}] {role}: {msg.content[:80]}")

print(f"\n  Messages in memory: {len(session_state['messages'])}")
print(f"  Last result: {session_state['last_result']}")


## How MessagesState Memory Works

```python
# Turn 1:
state["messages"] = [HumanMessage("what is 8 plus 7?")]
# Agent runs, returns AIMessage("15. RESULT: 15")
# add_messages reducer appends → messages = [Human(...), AI("15...")]
# last_result = 15.0

# Turn 2:
state["messages"] = [..., HumanMessage("multiply that by 2")]
# Agent sees full history + last_result=15
# Answers: "15 × 2 = 30. RESULT: 30"
# messages = [Human, AI, Human, AI]
# last_result = 30.0
```

### The `add_messages` Reducer
```python
# Defined in: langgraph.graph.message
# Behavior:
# 1. Appends new messages to existing list
# 2. Deduplicates by message.id (prevents issues with re-runs)
# 3. Handles RemoveMessage to delete specific messages
```

## Summary

| | Lesson 12 | Lesson 13 |
|--|-----------|-----------|
| Memory | None (stateless) | Short-term (in-memory) |
| Multi-turn | No | Yes |
| Context | None | Full conversation history |

## Limitations → What Lesson 14 Solves
- ❌ Memory grows unboundedly — after 100 turns, the LLM context window overflows
- ❌ Sending 1000 old messages to every LLM call is wasteful and expensive
- ❌ **Lesson 14**: Message trimming and summarization to keep context manageable
